# **SN_AI – Predicting Serie A Match Results Using RNNs**

---

## Introduction / Context
The **SN_AI** project aims to predict Serie A match results from the 2015/2016 season to the ongoing 2025/2026 season.  
We will use **Recurrent Neural Networks (RNNs)** to capture temporal dependencies in historical team data and generate probabilistic match outcome predictions.

**Note:** Match results depend on more than historical results. Real-life factors such as player fitness, absences, strategies, and transfer market dynamics cannot currently be modeled due to insufficient data.

---

## Dataset and Split
- **Dataset:** each match is represented as a JSON object containing date, teams, score, and match number.  

```json
{
  "MatchNumber": 1,
  "RoundNumber": 1,
  "DateUtc": "2017-08-19 16:00:00Z",
  "Location": "Juventus Stadium",
  "HomeTeam": "Juventus",
  "AwayTeam": "Cagliari",
  "Group": null,
  "HomeTeamScore": 3,
  "AwayTeamScore": 0
}
```

**The dataset is divided in this way:**
- 70% for training
- 15% for validation
- 15% for testing

## Features
Main features used for the RNN:

- **Teams:** `home_team_name`, `away_team_name`  
- **Recent performance (last 5 matches):**  
  `home_last5_points`, `away_last5_points`  
  `home_last5_goals`, `away_last5_goals`  
- **Season averages:**  
  `home_avg_goals_scored`, `away_avg_goals_scored`  
  `home_avg_goals_conceded`, `away_avg_goals_conceded`  
- **Elo ratings:**  
  `elo_home_team`, `elo_away_team`, `elo_diff`  
- **Recent performance differences:**  
  `goal_diff_last5`, `points_diff_last5`  
- **Match history last 2 seasons:**  
  `last2yrs_match_history`  

---

## RNN Output
The network should output a Python object:

```python
result_predicted = {
    "home_win_prob": ...,
    "away_win_prob": ...,
    "draw_prob": ...,
    "home_goal_prediction": ...,
    "away_goal_prediction": ...,
    "over_2_5_prob": ...,
    "under_2_5_prob": ...
}
```

## Architecture and Method
- Network type: **RNN** (to capture temporal dependencies)  
- Activation function: **ReLU**  
- Loss function: **cross-entropy** (for result probabilities) and **MSE** (for goals)  
- Backpropagation: **BPTT (Backpropagation Through Time)**  
- Optimizer: **Adam**  
- Regularization: **dropout and L2 weight decay**  
- Validation: train/validation/test as indicated

---

## Evaluation
- Metrics:
  - Accuracy / F1-score / Precision / Recall (for win/draw/loss probabilities)  
  - MAE / MSE (for predicted goals)  
  - Confusion matrix  
- Feature importance analysis to understand which features most influence predictions

---

In [225]:
from pathlib import Path

def read_file(file_path: Path) -> str:
    """Read and return the content of a file."""
    print(f"Reading file: {file_path}")
    return file_path.read_text(encoding="utf-8")


def read_folder_files(folder_path: str) -> dict:
    """Read all files in a folder and return a dict {filename: content}."""
    folder = Path(folder_path)

    return {
        file.name: read_file(file)
        for file in folder.iterdir()
        if file.is_file()
    }

############################################################################################################################################################

files_path= "./json_files"
file_contents = read_folder_files(files_path)

Reading file: json_files/SerieA_0405.json
Reading file: json_files/SerieA_9798.json
Reading file: json_files/SerieA_0203.json
Reading file: json_files/SerieA_1516.json
Reading file: json_files/SerieA_1920.json
Reading file: json_files/SerieA_2526.json
Reading file: json_files/SerieA_0809.json
Reading file: json_files/SerieA_9900.json
Reading file: json_files/SerieA_0506.json
Reading file: json_files/SerieA_0910.json
Reading file: json_files/SerieA_2223.json
Reading file: json_files/SerieA_1819.json
Reading file: json_files/SerieA_2425.json
Reading file: json_files/SerieA_9697.json
Reading file: json_files/SerieA_0708.json
Reading file: json_files/SerieA_2324.json
Reading file: json_files/SerieA_9495.json
Reading file: json_files/SerieA_2021.json
Reading file: json_files/SerieA_9899.json
Reading file: json_files/SerieA_1011.json
Reading file: json_files/SerieA_0102.json
Reading file: json_files/SerieA_1617.json
Reading file: json_files/SerieA_2122.json
Reading file: json_files/SerieA_17

In [226]:
import json
from collections import defaultdict
from datetime import datetime
from enum import IntEnum
import math


def create_team_enum(teams_dict):
    enum_dict = {
        team.upper().replace(" ", "_"): i
        for i, team in enumerate(teams_dict.keys())
    }
    return IntEnum("SerieATeams", enum_dict)


def attach_enum_to_teams(teams_dict, enum_cls):
    return {
        team: (enum_cls[team.upper().replace(" ", "_")], points)
        for team, points in teams_dict.items()
    }


def clean_match(match: dict) -> dict:
    """Remove unnecessary keys from match."""
    keys_to_remove = {"Location", "Group"}
    return {k: v for k, v in match.items() if k not in keys_to_remove}


def update_team_points(match: dict, teams: dict):
    """Update cumulative team points based on match result."""
    home = match["HomeTeam"]
    away = match["AwayTeam"]
    home_score = match["HomeTeamScore"]
    away_score = match["AwayTeamScore"]

    teams.setdefault(home, 0)
    teams.setdefault(away, 0)

    if home_score > away_score:
        teams[home] += 3
    elif home_score < away_score:
        teams[away] += 3
    else:
        teams[home] += 1
        teams[away] += 1


def track_past_match(match: dict, past_matches: dict, season_number: str):
    """
    Store past match results using a tuple key: (home_team, away_team).
    Save numeric goals and timestamp in milliseconds.
    """
    home = match["HomeTeam"]
    away = match["AwayTeam"]
    home_score = match["HomeTeamScore"]
    away_score = match["AwayTeamScore"]

    key = (home, away)
    match_day_str = match["DateUtc"]  # es: "2015-08-22 18:00:00Z"

    match_dt = datetime.strptime(match_day_str, "%Y-%m-%d %H:%M:%SZ")
    match_day_m = int(match_dt.timestamp() * 1000)

    past_matches.setdefault(key, [])
    past_matches[key].append({
        "home_goals": home_score,
        "away_goals": away_score,
        "matchDay": match_day_m,
        "season": season_number
    })


def parse_json(json_content: str, teams: dict, past_matches: dict, season_number: str):
    """Parse JSON content and update statistics."""
    try:
        matches = json.loads(json_content)

        if not isinstance(matches, list):
            print("Expected a list of matches")
            return []

        cleaned_matches = []

        for match in matches:
            match = clean_match(match)

            if match["HomeTeamScore"] is not None and match["AwayTeamScore"] is not None:
                update_team_points(match, teams)
                track_past_match(match, past_matches, season_number)

            cleaned_matches.append(match)

        return cleaned_matches

    except json.JSONDecodeError as e:
        print(f"Error parsing JSON: {e}")
        return []


def parse_dataset(file_contents: dict, teams: dict, past_matches: dict):
    """Parse all files in a dataset."""
    parsed_contents = {}

    for file, content in file_contents.items():
        season_number = file.split("_")[1].split(".")[0]
        print(f"Parsing file: {file} (Season {season_number})")
        parsed_contents[file] = parse_json(content, teams, past_matches, season_number)

    return parsed_contents


def attach_base_elo_to_teams(teams_dict):
    """Attach base ELO rating to teams."""
    base_elo = 1500
    teams_dict_with_elo = {}

    for team, (enum_val, points) in teams_dict.items():
        teams_dict_with_elo[team] = (enum_val, points, base_elo)

    return teams_dict_with_elo


def goal_diff_multiplier(goal_diff):
    return math.log(goal_diff + 1)


def update_elo(home_rating, away_rating, home_goals, away_goals, K=20, home_advantage=80):
    if home_goals > away_goals:
        S_home, S_away = 1, 0
    elif home_goals < away_goals:
        S_home, S_away = 0, 1
    else:
        S_home, S_away = 0.5, 0.5

    E_home = 1 / (1 + 10 ** ((away_rating - (home_rating + home_advantage)) / 400))
    E_away = 1 / (1 + 10 ** ((home_rating - (away_rating - home_advantage)) / 400))

    goal_diff = abs(home_goals - away_goals)
    multiplier = goal_diff_multiplier(goal_diff) if goal_diff > 0 else 1

    new_home = home_rating + K * multiplier * (S_home - E_home)
    new_away = away_rating + K * multiplier * (S_away - E_away)

    return new_home, new_away


#####################################################################
# STRUCTURES
#####################################################################

all_possible_past_matches = {}
all_possible_teams = {}

train_parsed_contents = parse_dataset(
    file_contents, all_possible_teams, all_possible_past_matches
)

print(f"\nTotal matches parsed: {sum(len(matches) for matches in all_possible_past_matches.values())}")
print(f"Total teams parsed: {len(all_possible_teams)}")

SerieATeams = create_team_enum(all_possible_teams)
num_teams = len(SerieATeams)

all_possible_teams = attach_enum_to_teams(all_possible_teams, SerieATeams)
all_possible_teams = attach_base_elo_to_teams(all_possible_teams)

#####################################################################
# SORT PAST MATCHES BY matchDay
#####################################################################

for match_key, match_list in all_possible_past_matches.items():
    match_list.sort(key=lambda x: x["matchDay"])

for match_key, match_list in all_possible_past_matches.items():
    match_days = [match["matchDay"] for match in match_list]
    if match_days != sorted(match_days):
        print(f"Error: Match list for {match_key} is not sorted by matchDay.")

#####################################################################
# BUILD team_matches
#####################################################################

team_matches = {team: [] for team in all_possible_teams.keys()}

for match_key, match_list in all_possible_past_matches.items():
    home_team, away_team = match_key  # <-- qui non c'è più split("-")

    for match in match_list:
        # record dal punto di vista della squadra di casa
        team_matches[home_team].append({
            "home_team": home_team,
            "away_team": away_team,
            "is_home": True,
            "home_goals": match["home_goals"],
            "away_goals": match["away_goals"],
            "matchDay": match["matchDay"],
            "season": match["season"]
        })

        # record dal punto di vista della squadra in trasferta
        team_matches[away_team].append({
            "home_team": home_team,
            "away_team": away_team,
            "is_home": False,
            "home_goals": match["home_goals"],
            "away_goals": match["away_goals"],
            "matchDay": match["matchDay"],
            "season": match["season"]
        })

#####################################################################
# SORT team_matches BY matchDay
#####################################################################

for team, matches in team_matches.items():
    matches.sort(key=lambda x: x["matchDay"])

for team, matches in team_matches.items():
    match_days = [match["matchDay"] for match in matches]
    if match_days != sorted(match_days):
        print(f"Error: Match list for team {team} is not sorted by matchDay.")

# divide all_possible_past_matches in train, eval and test based on matchDay and creating dictionaries, train 70%, eval 15% and test 15%
train_data = {}
eval_data = {}
test_data = {}

for match_key, matches in all_possible_past_matches.items():
    for match in matches:
        match_day = match["matchDay"]
        if match_day < 0.7 * max(match["matchDay"] for matches in all_possible_past_matches.values() for match in matches):
            train_data.setdefault(match_key, []).append(match)
        elif match_day < 0.85 * max(match["matchDay"] for matches in all_possible_past_matches.values() for match in matches):
            eval_data.setdefault(match_key, []).append(match)
        else:
            test_data.setdefault(match_key, []).append(match)

# group train_data, eval_data and test_data by match_key
train_matches = {match_key: matches for match_key, matches in train_data.items() if matches}
eval_matches = {match_key: matches for match_key, matches in eval_data.items() if matches}
test_matches = {match_key: matches for match_key, matches in test_data.items() if matches}

Parsing file: SerieA_0405.json (Season 0405)
Parsing file: SerieA_9798.json (Season 9798)
Parsing file: SerieA_0203.json (Season 0203)
Parsing file: SerieA_1516.json (Season 1516)
Parsing file: SerieA_1920.json (Season 1920)
Parsing file: SerieA_2526.json (Season 2526)
Parsing file: SerieA_0809.json (Season 0809)
Parsing file: SerieA_9900.json (Season 9900)
Parsing file: SerieA_0506.json (Season 0506)
Parsing file: SerieA_0910.json (Season 0910)
Parsing file: SerieA_2223.json (Season 2223)
Parsing file: SerieA_1819.json (Season 1819)
Parsing file: SerieA_2425.json (Season 2425)
Parsing file: SerieA_9697.json (Season 9697)
Parsing file: SerieA_0708.json (Season 0708)
Parsing file: SerieA_2324.json (Season 2324)
Parsing file: SerieA_9495.json (Season 9495)
Parsing file: SerieA_2021.json (Season 2021)
Parsing file: SerieA_9899.json (Season 9899)
Parsing file: SerieA_1011.json (Season 1011)
Parsing file: SerieA_0102.json (Season 0102)
Parsing file: SerieA_1617.json (Season 1617)
Parsing fi

In [227]:
# =========================================================
# FEATURE ENGINEERING
# =========================================================

def find_previous5_team_matches(team, matchday, team_matches):
    matches = team_matches.get(team, [])
    previous_matches = [m for m in matches if m["matchDay"] < matchday]
    return previous_matches[-5:]


def previous5_points(team, previous5_matches):
    points = 0
    for match in previous5_matches:
        home_team = match["home_team"]
        away_team = match["away_team"]
        home_score = match["home_goals"]
        away_score = match["away_goals"]

        if team == home_team:
            if home_score > away_score:
                points += 3
            elif home_score == away_score:
                points += 1
        elif team == away_team:
            if away_score > home_score:
                points += 3
            elif away_score == home_score:
                points += 1
    return points


def previous5_goals(team, previous5_matches):
    goals = 0
    for match in previous5_matches:
        home_team = match["home_team"]
        away_team = match["away_team"]
        home_score = match["home_goals"]
        away_score = match["away_goals"]

        if team == home_team:
            goals += int(home_score)
        elif team == away_team:
            goals += int(away_score)
    return goals


def season_avg_goals_scored(team, matchday, season, team_matches):
    goals = 0
    matches_count = 0
    matches = team_matches.get(team, [])

    for match in matches:
        if match["season"] == season and match["matchDay"] < matchday:
            home_team = match["home_team"]
            away_team = match["away_team"]
            home_score = match["home_goals"]
            away_score = match["away_goals"]

            if team == home_team:
                goals += int(home_score)
                matches_count += 1
            elif team == away_team:
                goals += int(away_score)
                matches_count += 1

    return round(goals / matches_count, 2) if matches_count > 0 else 0.0


def season_avg_goals_conceded(team, matchday, season, team_matches):
    goals_conceded = 0
    matches_count = 0
    matches = team_matches.get(team, [])

    for match in matches:
        if match["season"] == season and match["matchDay"] < matchday:
            home_team = match["home_team"]
            away_team = match["away_team"]
            home_score = match["home_goals"]
            away_score = match["away_goals"]

            if team == home_team:
                goals_conceded += int(away_score)
                matches_count += 1
            elif team == away_team:
                goals_conceded += int(home_score)
                matches_count += 1

    return round(goals_conceded / matches_count, 2) if matches_count > 0 else 0.0


def previous5_goal_diff(team, previous5_matches):
    goal_diff = 0
    for match in previous5_matches:
        home_team = match["home_team"]
        away_team = match["away_team"]
        home_score = match["home_goals"]
        away_score = match["away_goals"]

        if team == home_team:
            goal_diff += int(home_score) - int(away_score)
        elif team == away_team:
            goal_diff += int(away_score) - int(home_score)

    return goal_diff


def previous5_points_mean(team, previous5_matches):
    return round(previous5_points(team, previous5_matches) / len(previous5_matches), 2) if len(previous5_matches) > 0 else 0.0


def find_last2yrs_match_history(team1, team2, season, all_possible_past_matches):
    """
    Cerca gli scontri diretti tra team1 e team2 nelle 2 stagioni precedenti.
    all_possible_past_matches deve avere chiavi tuple: (home_team, away_team)
    """
    season_int = int(season)
    first = season_int // 100
    second = season_int % 100

    previous_season = f"{first-1:02d}{second-1:02d}"
    two_seasons_ago = f"{first-2:02d}{second-2:02d}"

    direct_matches = []
    possible_keys = [(team1, team2), (team2, team1)]

    for key in possible_keys:
        for match in all_possible_past_matches.get(key, []):
            if match["season"] in [previous_season, two_seasons_ago]:
                direct_matches.append(match)

    direct_matches.sort(key=lambda x: x["matchDay"])
    return direct_matches


def create_label(home_goals, away_goals):
    if home_goals > away_goals:
        return 0   # home win
    elif home_goals == away_goals:
        return 1   # draw
    else:
        return 2   # away win


# =========================================================
# ELO
# =========================================================

import math

def update_elo(home_rating, away_rating, home_goals, away_goals, k=20, home_advantage=80):
    # actual result
    if home_goals > away_goals:
        s_home = 1.0
        s_away = 0.0
    elif home_goals < away_goals:
        s_home = 0.0
        s_away = 1.0
    else:
        s_home = 0.5
        s_away = 0.5

    # expected result
    adjusted_home = home_rating + home_advantage
    e_home = 1.0 / (1.0 + 10 ** ((away_rating - adjusted_home) / 400.0))
    e_away = 1.0 - e_home

    # margin multiplier
    goal_diff = abs(home_goals - away_goals)
    if goal_diff <= 1:
        margin_multiplier = 1.0
    elif goal_diff == 2:
        margin_multiplier = 1.5
    else:
        margin_multiplier = 1.75 + 0.1 * (goal_diff - 3)

    new_home = home_rating + k * margin_multiplier * (s_home - e_home)
    new_away = away_rating + k * margin_multiplier * (s_away - e_away)

    return new_home, new_away


# =========================================================
# BUILD FEATURES
# =========================================================

def build_features_and_labels(matches_dict, all_possible_teams, team_matches_map, current_elo_ratings=None):
    if current_elo_ratings is None:
        current_elo_ratings = {team: 1500.0 for team in all_possible_teams.keys()}
    else:
        current_elo_ratings = current_elo_ratings.copy()

    all_matches_flat = []
    for match_key, results in matches_dict.items():
        home_team, away_team = match_key
        for res in results:
            all_matches_flat.append({
                "home_team": home_team,
                "away_team": away_team,
                **res
            })

    all_matches_flat.sort(key=lambda x: x["matchDay"])

    features_teams_ids_list = []
    features_list = []
    labels_list = []

    for m in all_matches_flat:
        h_team = m["home_team"]
        a_team = m["away_team"]
        m_day = m["matchDay"]
        season_num = m["season"]

        elo_h_before = current_elo_ratings[h_team]
        elo_a_before = current_elo_ratings[a_team]

        h_prev = find_previous5_team_matches(h_team, m_day, team_matches_map)
        a_prev = find_previous5_team_matches(a_team, m_day, team_matches_map)

        features = [
            previous5_points(h_team, h_prev),
            previous5_points(a_team, a_prev),

            previous5_goals(h_team, h_prev),
            previous5_goals(a_team, a_prev),

            season_avg_goals_scored(h_team, m_day, season_num, team_matches_map),
            season_avg_goals_scored(a_team, m_day, season_num, team_matches_map),

            season_avg_goals_conceded(h_team, m_day, season_num, team_matches_map),
            season_avg_goals_conceded(a_team, m_day, season_num, team_matches_map),

            elo_h_before,
            elo_a_before,
            elo_h_before - elo_a_before,

            previous5_goal_diff(h_team, h_prev),
            previous5_goal_diff(a_team, a_prev),

            previous5_points_mean(h_team, h_prev),
            previous5_points_mean(a_team, a_prev),
        ]

        label = create_label(m["home_goals"], m["away_goals"])

        features_teams_ids_list.append([
            all_possible_teams[h_team][0],
            all_possible_teams[a_team][0]
        ])
        features_list.append(features)
        labels_list.append(label)

        new_h, new_a = update_elo(
            elo_h_before,
            elo_a_before,
            m["home_goals"],
            m["away_goals"]
        )
        current_elo_ratings[h_team] = new_h
        current_elo_ratings[a_team] = new_a

    return features_teams_ids_list, features_list, labels_list, current_elo_ratings


# =========================================================
# DATA PREPARATION
# =========================================================

from sklearn.preprocessing import StandardScaler
import numpy as np

scaler = StandardScaler()

initial_elo = {team: 1500.0 for team in all_possible_teams.keys()}

# TRAIN
train_team_ids, X_train_num, Y_train, elo_after_train = build_features_and_labels(
    train_matches,
    all_possible_teams,
    team_matches,
    current_elo_ratings=initial_elo
)

# EVAL
eval_team_ids, X_eval_num, Y_eval, elo_after_eval = build_features_and_labels(
    eval_matches,
    all_possible_teams,
    team_matches,
    current_elo_ratings=elo_after_train
)

# TEST
test_team_ids, X_test_num, Y_test, elo_after_test = build_features_and_labels(
    test_matches,
    all_possible_teams,
    team_matches,
    current_elo_ratings=elo_after_eval
)


# scale only numeric features
X_train_num = scaler.fit_transform(X_train_num)
X_eval_num = scaler.transform(X_eval_num)
X_test_num = scaler.transform(X_test_num)

# combine team IDs + numeric features
X_train = []
for i in range(len(X_train_num)):
    X_train.append(list(train_team_ids[i]) + list(X_train_num[i]))
print(f"Sample feature vector (train): {X_train[0]}")
# calculate total number of features
features_num = len(X_train[0])
print(f"Total numeric features: {features_num}")

X_eval = []
for i in range(len(X_eval_num)):
    X_eval.append(list(eval_team_ids[i]) + list(X_eval_num[i]))

X_test = []
for i in range(len(X_test_num)):
    X_test.append(list(test_team_ids[i]) + list(X_test_num[i]))


Sample feature vector (train): [<SerieATeams.ROMA: 16>, <SerieATeams.SAMPDORIA: 18>, np.float64(-2.0012015108195422), np.float64(-2.068312689090965), np.float64(-2.1578368886861217), np.float64(-2.1687948997010236), np.float64(-2.325703053737516), np.float64(-2.2707100883936526), np.float64(-2.4403662791127476), np.float64(-2.371024010535339), np.float64(-0.513265427105604), np.float64(-0.5223329640864217), np.float64(0.005327225823368849), np.float64(0.027442490880418214), np.float64(-0.04005485301064176), np.float64(-2.0101177798449097), np.float64(-2.0848360649761273)]
Total numeric features: 17


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class MatchOutcomePredictor(nn.Module):
    def __init__(self, num_teams, num_features, num_classes=3, embedding_size=24):
        super(MatchOutcomePredictor, self).__init__()

        self.team_embedding = nn.Embedding(num_teams, embedding_size)

        # Le prime 2 colonne sono gli id delle squadre: home_team e away_team
        # Vengono sostituite dai rispettivi embedding
        input_size = (num_features - 2) + (2 * embedding_size)

        self.fc1 = nn.Linear(input_size, 128)
        self.bn1 = nn.BatchNorm1d(128)
        self.dropout1 = nn.Dropout(0.2)

        self.fc2 = nn.Linear(128, 64)
        self.bn2 = nn.BatchNorm1d(64)
        self.dropout2 = nn.Dropout(0.2)

        self.fc3 = nn.Linear(64, num_classes)

    def forward(self, x):
        home_team_embedded = self.team_embedding(x[:, 0].long())
        away_team_embedded = self.team_embedding(x[:, 1].long())

        x = torch.cat((home_team_embedded, away_team_embedded, x[:, 2:]), dim=1)

        x = self.dropout1(torch.relu(self.bn1(self.fc1(x))))
        x = self.dropout2(torch.relu(self.bn2(self.fc2(x))))
        x = self.fc3(x)
        return x

model = MatchOutcomePredictor(num_teams=num_teams, num_features=features_num)
print(model)

# Define a custom Dataset
class MatchDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


batch_size = 32

train_dataset = MatchDataset(X_train, Y_train)
eval_dataset = MatchDataset(X_eval, Y_eval)
test_dataset = MatchDataset(X_test, Y_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
eval_loader = DataLoader(eval_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

MatchOutcomePredictor(
  (team_embedding): Embedding(52, 24)
  (fc1): Linear(in_features=63, out_features=128, bias=True)
  (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout1): Dropout(p=0.2, inplace=False)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout2): Dropout(p=0.2, inplace=False)
  (fc3): Linear(in_features=64, out_features=3, bias=True)
)
(tensor([ 1.6000e+01,  1.8000e+01, -2.0012e+00, -2.0683e+00, -2.1578e+00,
        -2.1688e+00, -2.3257e+00, -2.2707e+00, -2.4404e+00, -2.3710e+00,
        -5.1327e-01, -5.2233e-01,  5.3272e-03,  2.7442e-02, -4.0055e-02,
        -2.0101e+00, -2.0848e+00]), tensor(0))


In [229]:
import numpy as np
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Conta quante volte compare ogni classe nel train
class_counts = np.bincount(Y_train, minlength=3)

# Evita divisioni per zero
class_counts = np.where(class_counts == 0, 1, class_counts)

# Peso inversamente proporzionale alla frequenza
class_weights = len(Y_train) / (3 * class_counts)

class_weights = torch.tensor(class_weights, dtype=torch.float32, device=device)

print("Class counts:", class_counts)
print("Class weights:", class_weights)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5) #  1e-4 or 1e-5
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

Class counts: [2329 1413 1146]
Class weights: tensor([0.6996, 1.1531, 1.4218])


In [230]:
from sklearn.metrics import accuracy_score, classification_report
import copy

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0

    for X_batch, y_batch in dataloader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X_batch.size(0)

    epoch_loss = running_loss / len(dataloader.dataset)
    return epoch_loss

def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0

    all_preds = []
    all_targets = []

    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            running_loss += loss.item() * X_batch.size(0)

            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(y_batch.cpu().numpy())

    epoch_loss = running_loss / len(dataloader.dataset)
    acc = accuracy_score(all_targets, all_preds)

    return epoch_loss, acc, all_preds, all_targets

num_epochs = 50
best_eval_acc = 0.0
best_state_dict = None

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    eval_loss, eval_acc, _, _ = evaluate(model, eval_loader, criterion, device)
    scheduler.step(eval_loss)
    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Eval Loss: {eval_loss:.4f} | "
        f"Eval Accuracy: {eval_acc:.4f} | "
    )

    
    if eval_acc >= best_eval_acc:
        best_eval_acc = eval_acc
        best_state_dict = copy.deepcopy(model.state_dict())

model.load_state_dict(best_state_dict)

test_loss, test_acc, test_preds, test_targets = evaluate(model, test_loader, criterion, device)

print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}\n")

print(classification_report(
    test_targets,
    test_preds,
    target_names=["Home Win", "Draw", "Away Win"]
))

# Spiegazione delle metriche in italiano ed in modo semplice:
print("Spiegazione delle metriche:")
print("\t- Precision: La precision misura la percentuale di predizioni corrette per una classe specifica rispetto a tutte le predizioni fatte per quella classe. Ad esempio, se il modello predice 'Home Win' 100 volte e 80 di queste sono corrette, la precisione per 'Home Win' è dell'80%.")
print("\t- Recall: La recall misura la percentuale di elementi di una classe specifica che sono stati correttamente identificati dal modello rispetto a tutti gli elementi effettivi di quella classe. Ad esempio, se ci sono 100 match che sono 'Home Win' e il modello ne identifica 80, la recall per 'Home Win' è dell'80%.")
print("\t- F1-Score: L'F1-Score è la media armonica della precisione e della recall, fornendo una singola metrica che bilancia entrambe. È utile quando si vuole considerare sia la precisione che la recall in una sola misura.")

Epoch 1/50 | Train Loss: 1.0677 | Eval Loss: 1.0297 | Eval Accuracy: 0.5060 | 
Epoch 2/50 | Train Loss: 1.0312 | Eval Loss: 1.0303 | Eval Accuracy: 0.5054 | 
Epoch 3/50 | Train Loss: 1.0184 | Eval Loss: 1.0263 | Eval Accuracy: 0.4910 | 
Epoch 4/50 | Train Loss: 1.0090 | Eval Loss: 1.0308 | Eval Accuracy: 0.4922 | 
Epoch 5/50 | Train Loss: 1.0007 | Eval Loss: 1.0358 | Eval Accuracy: 0.4976 | 
Epoch 6/50 | Train Loss: 0.9891 | Eval Loss: 1.0474 | Eval Accuracy: 0.4801 | 
Epoch 7/50 | Train Loss: 0.9838 | Eval Loss: 1.0469 | Eval Accuracy: 0.4898 | 
Epoch 8/50 | Train Loss: 0.9809 | Eval Loss: 1.0515 | Eval Accuracy: 0.4795 | 
Epoch 9/50 | Train Loss: 0.9733 | Eval Loss: 1.0548 | Eval Accuracy: 0.4657 | 
Epoch 10/50 | Train Loss: 0.9635 | Eval Loss: 1.0509 | Eval Accuracy: 0.4843 | 
Epoch 11/50 | Train Loss: 0.9452 | Eval Loss: 1.0599 | Eval Accuracy: 0.4723 | 
Epoch 12/50 | Train Loss: 0.9472 | Eval Loss: 1.0690 | Eval Accuracy: 0.4663 | 
Epoch 13/50 | Train Loss: 0.9387 | Eval Loss: 1.0